In [1]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

Setup done.

# 🗺️ Монады в Haskell — Полный справочник

> Монада — это тройка (M, return, >>=) удовлетворяющая трём законам.
> Но за этим сухим определением скрываются десятки конкретных паттернов.

Каждая монада в этом ноутбуке описана по схеме:
- 📐 **Реализация** — как устроена внутри
- ⭐ **Главные фичи** и вспомогательные функции
- 🛠️ **Примеры использования** — в каких ситуациях применять
- 🔭 **Категорная точка зрения**

---
📌 Содержание

| # | Монада | Суть |
|---|--------|------|
| 1 | `Maybe` | Вычисление, которое может провалиться |
| 2 | `Either e` | Провал с информацией об ошибке |
| 3 | `[]` List | Недетерминизм, множество результатов |
| 4 | `IO` | Побочные эффекты |
| 5 | `State s` | Изменяемое состояние |
| 6 | `Reader r` | Разделяемое окружение / конфиг |
| 7 | `Writer w` | Накопление лога |
| 8 | `Identity` | Тривиальная монада |
| 9 | `Cont r` | Продолжения (CPS) |
| 10 | `ST s` | Безопасная мутабельность |
| 11 | `STM` | Транзакционная память |
| 12 | `Parser` | Самодельный парсер-монада |
| 13 | `Free f` | Свободная монада / DSL |
| 14 | `Q` | Монада цитирования (Template Haskell) |
| 15 | **🔗 Связи** | Изоморфизмы, иерархия, таблица выбора |

---
> 💡 **Трансформеры монад** (`MaybeT`, `ExceptT`, `StateT`, `ReaderT`, `WriterT`, `ContT`) — отдельный ноутбук.

> **📦 Зависимости**
> **Пакеты:** `array`, `base`, `containers`, `mtl`, `stm`, `template-haskell`, `time`, `transformers`
> **Расширения:** `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


## Иерархия классов: Functor → Applicative → Monad

Каждый следующий класс **расширяет** предыдущий -- каждая Monad является Applicative, каждый Applicative -- Functor.

```
Functor f         fmap :: (a -> b) -> f a -> f b
   |
   v
Applicative f     pure  :: a -> f a
                  (<*>) :: f (a -> b) -> f a -> f b
   |
   v
Monad m           return :: a -> m a
                  (>>=)  :: m a -> (a -> m b) -> m b
                  join   :: m (m a) -> m a
```

### Законы на каждом уровне

**Functor** (2 закона):
```haskell
-- 1. Тождество: fmap id = id
-- 2. Композиция: fmap (f . g) = fmap f . fmap g
```

**Applicative** (4 закона):
```haskell
-- 1. Тождество:    pure id <*> v = v
-- 2. Композиция: pure (.) <*> u <*> v <*> w = u <*> (v <*> w)
-- 3. Гомоморфизм: pure f <*> pure x = pure (f x)
-- 4. Перестановка: u <*> pure y = pure ($ y) <*> u
```

**Monad** (3 закона):
```haskell
-- 1. Левая единица:  return a >>= f  =  f a
-- 2. Правая единица: m >>= return   =  m
-- 3. Ассоциативн:ость: (m >>= f) >>= g = m >>= (\x -> f x >>= g)
```

**Совместимость** Applicative и Monad:
```haskell
pure   = return
f <*> x = f >>= (\g -> x >>= (\y -> return (g y)))
fmap f x = x >>= (return . f)
```


In [2]:
-- Demonstrate Functor -> Applicative -> Monad hierarchy
-- fmap is the foundation
let xs = [1,2,3] :: [Int]
let demo_functor = fmap (*2) xs
putStrLn $ "fmap (*2) [1,2,3] = " ++ show demo_functor

-- Applicative adds pure and <*>
let fs = [(+1), (*2), (^2)] :: [Int -> Int]
let demo_applic = fs <*> [3,4]
putStrLn $ "[(+1),(*2),(^2)] <*> [3,4] = " ++ show demo_applic

-- Monad adds >>= (bind)
let safeRecip :: Double -> Maybe Double
    safeRecip x = if x == 0 then Nothing else Just (1/x)
let safeLog :: Double -> Maybe Double
    safeLog x = if x <= 0 then Nothing else Just (log x)
let demo_monad1 = Just 4.0 >>= safeRecip >>= safeLog
let demo_monad2 = Just 0.0 >>= safeRecip >>= safeLog
putStrLn $ "Just 4.0 >>= safeRecip >>= safeLog = " ++ show demo_monad1
putStrLn $ "Just 0.0 >>= safeRecip >>= safeLog = " ++ show demo_monad2

-- Monad vs Applicative: bind can depend on previous value
let monad_dep = Just 3 >>= (\n -> Just (replicate n "x"))
putStrLn $ "Just 3 >>= replicate: " ++ show monad_dep
-- Applicative cannot: structure fixed ahead of time
let applic_indep = pure replicate <*> Just 3 <*> Just "x"
putStrLn $ "pure replicate <*> Just 3 <*> Just x = " ++ show applic_indep

-- join :: m (m a) -> m a
let nested = Just (Just 42)
let joined = nested >>= id
putStrLn $ "join (Just (Just 42)) = " ++ show joined


fmap (*2) [1,2,3] = [2,4,6]

[(+1),(*2),(^2)] <*> [3,4] = [4,5,6,8,9,16]

Just 4.0 >>= safeRecip >>= safeLog = Just (-1.3862943611198906)

Just 0.0 >>= safeRecip >>= safeLog = Nothing

Just 3 >>= replicate: Just ["x","x","x"]

pure replicate <*> Just 3 <*> Just x = Just ["x","x","x"]

join (Just (Just 42)) = Just 42

## Коммутативная диаграмма монады

Монада -- это **моноид в категории эндофункторов** `Hask^Hask`.
Тройка `(M, η, μ)` где:
- `M : Hask -> Hask` -- эндофунктор
- `η : Id -> M` -- естественное преобразование (`return`), единица
- `μ : M² -> M` -- умножение (`join`), склейка

### Треугольники единицы (Left/Right Identity)

```
       eta_M        mu
M ---------> M^2 -------> M
|            |            |
id           |           id
|            v            |
M ========== M ========== M

       M_eta        mu
M ---------> M^2 -------> M
(оба треугольника коммутативны)
```

В Haskell: `join . return = id` и `join . fmap return = id`

### Пятиугольник ассоциативности

```
           mu_M
M^3 ---------> M^2
 |              |
M_mu           mu
 |              |
 v              v
M^2 ---------> M
           mu
(квадрат коммутативен)
```

В Haskell: `join . join = join . fmap join`

### Связь с >>=

```haskell
-- join через >>=
join m = m >>= id

-- >>= через join
m >>= f = join (fmap f m)
```

Оба определения **эквивалентны**: законы монады через >>= и через join/eta/mu -- одно и то же.


In [3]:
-- Verify join/eta monad laws for Maybe
import Control.Monad (join)

-- Law 1: join . return = id (for Maybe)
let law1_test :: Maybe Int -> Bool
    law1_test m = (join . return) m == m
putStrLn $ "join . return = id: " ++ show (all law1_test [Nothing, Just 1, Just 42])

-- Law 2: join . fmap return = id
let law2_test :: Maybe Int -> Bool
    law2_test m = (join . fmap return) m == m
putStrLn $ "join . fmap return = id: " ++ show (all law2_test [Nothing, Just 1, Just 42])

-- Law 3: join . join = join . fmap join (needs 3-level nesting)
let mmm :: [Maybe (Maybe (Maybe Int))]
    mmm = [Nothing, Just Nothing, Just (Just Nothing), Just (Just (Just 42))]
let law3_test = map (join . join) mmm == map (join . fmap join) mmm
putStrLn $ "join . join = join . fmap join: " ++ show law3_test

-- m >>= f = join (fmap f m)
let safeDiv :: Int -> Int -> Maybe Int
    safeDiv _ 0 = Nothing
    safeDiv x y = Just (x `div` y)
let via_bind = Just 10 >>= safeDiv 100
let via_join = join (fmap (safeDiv 100) (Just 10))
putStrLn $ "(>>=) vs join.fmap: " ++ show via_bind ++ " == " ++ show via_join ++ " -> " ++ show (via_bind == via_join)


join . return = id: True

join . fmap return = id: True

join . join = join . fmap join: True

(>>=) vs join.fmap: Just 10 == Just 10 -> True

## Что делает каждая монада: схема до/после bind

Каждая монада добавляет **неявный контекст** к обычным функциям `a -> b`:

```
Maybe     a --> b   С возможным отсутствием (Nothing)
          |
          v
          a --> Maybe b    (a -> Maybe b)

Either e  a --> b   С возможной ошибкой e
          |
          v
          a --> Either e b  (a -> Either e b)

[]        a --> b   Недетерминизм: несколько ответов
          |
          v
          a --> [b]         (a -> [b])

State s   a --> b   Читает/пишет состояние s
          |
          v
          a --> s -> (b, s)  (a -> State s b)

Reader r  a --> b   Читает окружение r (только чтение)
          |
          v
          a --> r -> b      (a -> Reader r b)

Writer w  a --> b   Пишет в журнал w (моноид)
          |
          v
          a --> (b, w)      (a -> Writer w b)

Cont r    a --> b   Инверсия управления (continuation passing)
          |
          v
          a --> (b -> r) -> r  (a -> Cont r b)
```

### Клейсли-стрелки (Kleisli arrows)

Бинд `>>=` -- это **композиция стрелок Клейсли**:
```haskell
(>=>) :: Monad m => (a -> m b) -> (b -> m c) -> (a -> m c)
f >=> g = \x -> f x >>= g
```
Каждая монада образует **категорию Клейсли**: объекты -- те же типы, стрелки `a -> b` заменяются на `a -> m b`.


In [4]:
import Control.Monad (forM_)
import Control.Monad.State
import Control.Monad.Reader
import Control.Monad.Writer

-- Kleisli composition operator
let (>=>) :: Monad m => (a -> m b) -> (b -> m c) -> a -> m c
    f >=> g = \x -> f x >>= g

-- Maybe: safe arithmetic pipeline
let safeSqrt :: Double -> Maybe Double
    safeSqrt x = if x < 0 then Nothing else Just (sqrt x)
let safeLog2 :: Double -> Maybe Double
    safeLog2 x = if x <= 0 then Nothing else Just (log x / log 2)
let pipeline = safeSqrt >=> safeLog2
putStrLn "--- Maybe Kleisli ---"
putStrLn $ "pipeline 16.0 = " ++ show (pipeline 16.0)
putStrLn $ "pipeline (-1) = " ++ show (pipeline (-1))

-- Either: validation pipeline
let checkPos :: Int -> Either String Int
    checkPos x = if x > 0 then Right x else Left "must be positive"
let checkSmall :: Int -> Either String Int
    checkSmall x = if x < 1000 then Right x else Left "too large"
let validatePipeline = checkPos >=> checkSmall
putStrLn "--- Either Kleisli ---"
putStrLn $ "validate 42 = " ++ show (validatePipeline 42)
putStrLn $ "validate (-5) = " ++ show (validatePipeline (-5))
putStrLn $ "validate 9999 = " ++ show (validatePipeline 9999)

-- State: counter
let tick :: String -> State Int String
    tick label = do
        n <- get
        put (n + 1)
        return (label ++ "=" ++ show n)
let stateDemo = do
        a <- tick "first"
        b <- tick "second"
        c <- tick "third"
        return [a, b, c]
let (results, finalState) = runState stateDemo 0
putStrLn "--- State ---"
putStrLn $ "results: " ++ show results
putStrLn $ "final counter: " ++ show finalState

-- Reader: config lookup
let getHost :: Reader (String, Int) String
    getHost = asks fst
let getPort :: Reader (String, Int) Int
    getPort = asks snd
let mkUrl :: Reader (String, Int) String
    mkUrl = do
        h <- getHost
        p <- getPort
        return ("http://" ++ h ++ ":" ++ show p)
putStrLn "--- Reader ---"
putStrLn $ runReader mkUrl ("localhost", 8080)

-- Writer: logging
let loggedAdd :: Int -> Int -> Writer [String] Int
    loggedAdd x y = do
        tell ["adding " ++ show x ++ " + " ++ show y]
        return (x + y)
let computation = do
        a <- loggedAdd 3 4
        b <- loggedAdd a 10
        return b
let (value, logs) = runWriter computation
putStrLn "--- Writer ---"
putStrLn $ "value: " ++ show value
putStrLn $ "log: " ++ show logs


--- Maybe Kleisli ---

pipeline 16.0 = Just 2.0

pipeline (-1) = Nothing

--- Either Kleisli ---

validate 42 = Right 42

...

--- Reader ---

http://localhost:8080

--- Writer ---

value: 17

log: ["adding 3 + 4","adding 7 + 10"]

---
# 1️⃣ Maybe — «Вычисление, которое может провалиться»

## 📐 Реализация

```haskell
data Maybe a = Nothing | Just a

instance Functor Maybe where
  fmap _ Nothing  = Nothing
  fmap f (Just x) = Just (f x)

instance Applicative Maybe where
  pure = Just
  Nothing <*> _       = Nothing
  _       <*> Nothing = Nothing
  Just f  <*> Just x  = Just (f x)

instance Monad Maybe where
  return = pure
  Nothing >>= _ = Nothing   -- провал распространяется
  Just x  >>= f = f x       -- передаём значение дальше
```

Ключевая идея: **Nothing — это «короткое замыкание»**.
Как только один шаг вернул `Nothing` — вся цепочка возвращает `Nothing`.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `maybe` | `b -> (a->b) -> Maybe a -> b` | Извлечь с дефолтом |
| `fromMaybe` | `a -> Maybe a -> a` | Извлечь или дефолт |
| `isJust` / `isNothing` | `Maybe a -> Bool` | Проверка |
| `fromJust` | `Maybe a -> a` | Небезопасное извлечение |
| `listToMaybe` | `[a] -> Maybe a` | Первый элемент или Nothing |
| `maybeToList` | `Maybe a -> [a]` | В список (0 или 1 элемент) |
| `catMaybes` | `[Maybe a] -> [a]` | Отфильтровать Nothing |
| `mapMaybe` | `(a -> Maybe b) -> [a] -> [b]` | map + фильтр |

In [5]:
import Data.Maybe

-- fromMaybe: безопасное извлечение с дефолтом
fromMaybe 0 (Just 42)   -- 42
-- fromMaybe 0 Nothing  -- 0

42

In [6]:
-- catMaybes: убрать все Nothing из списка
catMaybes [Just 1, Nothing, Just 3, Nothing, Just 5]

[1,3,5]

In [7]:
-- mapMaybe: map + автоматическая фильтрация Nothing
import Data.Maybe (mapMaybe)
import Text.Read (readMaybe)

-- Парсим список строк, игнорируя непарсируемые
mapMaybe (readMaybe :: String -> Maybe Int) ["1", "foo", "3", "bar", "5"]

[1,3,5]

## 🛠️ Примеры использования

**Когда применять:** безопасный доступ к данным, поиск, парсинг,
цепочка операций где любая может вернуть «ничего».

In [8]:
-- Пример 1: Безопасный доступ к вложенным структурам
import qualified Data.Map.Strict as Map

type DB = Map.Map String (Map.Map String String)

db :: DB
db = Map.fromList
  [ ("alice", Map.fromList [("email","alice@mail.com"),("city","Москва")])
  , ("bob",   Map.fromList [("email","bob@mail.com")])
  ]

-- Безопасное получение поля пользователя
getField :: String -> String -> Maybe String
getField user field = do
  userMap <- Map.lookup user db
  Map.lookup field userMap

getField "alice" "city"    -- Just "Москва"
-- getField "bob"   "city" -- Nothing (нет поля)
-- getField "charlie" "email" -- Nothing (нет пользователя)

Just "\1052\1086\1089\1082\1074\1072"

In [9]:
-- Пример 2: Валидация и вычисление
safeHead :: [a] -> Maybe a
safeHead []    = Nothing
safeHead (x:_) = Just x

safeSqrt :: Double -> Maybe Double
safeSqrt x
  | x < 0    = Nothing
  | otherwise = Just (sqrt x)

-- Цепочка: взять первый элемент, умножить, взять корень
pipeline :: [Double] -> Maybe Double
pipeline xs = do
  x <- safeHead xs
  let y = x - 10
  safeSqrt y

pipeline [14.0]    -- Just 2.0
-- pipeline [5.0]  -- Nothing (корень из отрицательного)
-- pipeline []     -- Nothing (пустой список)

Just 2.0

## 🔭 Категорная точка зрения

В теории категорий `Maybe` — это **свободный функтор добавления единицы**:

- Категория: множества и функции между ними
- **Объекты:** типы `a`
- **Стрелки:** `a -> Maybe b` — так называемые **Клейсли-стрелки**
- **Композиция Клейсли:** `(>=>) :: (a -> Maybe b) -> (b -> Maybe c) -> (a -> Maybe c)`
- `Nothing` — **абсорбирующий элемент** (ноль): `Nothing >>= f = Nothing`
- `Just` — это **η (unit/return)**: вложение категории Set в Maybe

`Maybe` изоморфна типу `Either () a` — левая сторона кодирует «ничего» как `()`.

Алгебраически: `Maybe a ≅ 1 + a` (сумма единицы и `a`).

In [10]:
-- Клейсли-композиция: >=>
import Control.Monad ((>=>))

safeDiv :: Int -> Int -> Maybe Int
safeDiv _ 0 = Nothing
safeDiv x y = Just (x `div` y)

safeRecip :: Int -> Maybe Double
safeRecip 0 = Nothing
safeRecip n = Just (1.0 / fromIntegral n)

-- Компонуем две Клейсли-стрелки в одну
pipeline2 :: Int -> Int -> Maybe Double
pipeline2 x y = safeDiv x y >>= safeRecip

-- Или через >=>
pipeline3 :: Int -> Maybe Double
pipeline3 = safeDiv 100 >=> safeRecip

pipeline3 4    -- Just 0.25
-- pipeline3 0 -- Nothing

Just 4.0e-2

---
# 2️⃣ Either e — «Провал с объяснением»

## 📐 Реализация

```haskell
data Either e a = Left e | Right a

instance Monad (Either e) where
  return = Right
  Left  e >>= _ = Left e    -- ошибка распространяется
  Right x >>= f = f x       -- передаём значение дальше
```

**Соглашение:** `Left` = ошибка/провал, `Right` = успех (right = правильный).
В отличие от `Maybe`, при провале мы знаем **почему** он случился.

## ⭐ Главные функции

| Функция | Описание |
|---------|----------|
| `either f g e` | Свернуть: `f` для Left, `g` для Right |
| `isLeft` / `isRight` | Проверить тег |
| `fromLeft d` / `fromRight d` | Извлечь с дефолтом |
| `partitionEithers` | Разбить `[Either]` на два списка |
| `lefts` / `rights` | Выбрать только Left/Right из списка |
| `bimap f g` | Применить функции к обоим сторонам |

In [11]:
import Data.Either (partitionEithers, lefts, rights)

results :: [Either String Int]
results = [Right 1, Left "ошибка A", Right 3, Left "ошибка B", Right 5]

-- Разделить на ошибки и успехи
let (errs, oks) = partitionEithers results
putStrLn $ "Ошибки: " ++ show errs
putStrLn $ "Успехи: " ++ show oks

Ошибки: ["\1086\1096\1080\1073\1082\1072 A","\1086\1096\1080\1073\1082\1072 B"]

Успехи: [1,3,5]

## 🛠️ Примеры использования

**Когда применять:** валидация, обработка ошибок в бизнес-логике,
парсинг с диагностикой, замена исключений.

In [12]:
-- Пример 1: Типизированные ошибки
data ValidationError
  = EmptyField  String
  | TooShort    String Int
  | InvalidChar String Char
  deriving (Show)

validateName :: String -> Either ValidationError String
validateName ""   = Left (EmptyField "name")
validateName name
  | length name < 2        = Left (TooShort "name" 2)
  | any (not . validChar) name = Left (InvalidChar "name" (head $ filter (not . validChar) name))
  | otherwise              = Right name
  where validChar c = c `elem` (['a'..'z'] ++ ['A'..'Z'] ++ " -")

validateAge :: Int -> Either ValidationError Int  -- используем тот же тип ошибки
validateAge n
  | n < 0    = Left (EmptyField "age (negative)")
  | n > 150  = Left (TooShort  "age (too large)" 0)
  | otherwise = Right n

validateName "Alice"
-- validateName ""
-- validateName "A"

Line 12: Hoist not
Found:
any (not . validChar)
Why not:
(not . all validChar)Line 12: Hoist not
Found:
any (not . validChar) name
Why not:
not (all validChar name)

Right "Alice"

In [13]:
-- Пример 2: Цепочка валидаций через do
data User = User { userName :: String, userAge :: Int } deriving (Show)

createUser :: String -> Int -> Either ValidationError User
createUser rawName rawAge = do
  name <- validateName rawName
  age  <- validateAge  rawAge
  return (User name age)

createUser "Alice" 30
-- createUser "" 30
-- createUser "Alice" (-5)

Right (User {userName = "Alice", userAge = 30})

In [14]:
-- Пример 3: Either как замена исключениям
import Control.Exception (try, SomeException, evaluate)

safeCompute :: Int -> Int -> Either String Double
safeCompute x y
  | y == 0    = Left "Деление на ноль"
  | x < 0    = Left "Отрицательный вход"
  | otherwise = Right (sqrt (fromIntegral x) / fromIntegral y)

-- Цепочка с накоплением контекста
withContext :: String -> Either String a -> Either String a
withContext ctx (Left e)  = Left (ctx ++ ": " ++ e)
withContext _   (Right x) = Right x

withContext "safeCompute(−1,2)" (safeCompute (-1) 2)

Left "safeCompute(\8722\&1,2): \1054\1090\1088\1080\1094\1072\1090\1077\1083\1100\1085\1099\1081 \1074\1093\1086\1076"

## 🔭 Категорная точка зрения

`Either e` — это **монада Клейсли для моноида ошибок**:

- Фиксируем тип ошибки `e` — он становится **параметром функтора**
- `Either e` — это **бифунктор**: `bimap :: (e->e') -> (a->b) -> Either e a -> Either e' b`
- Категория Клейсли: стрелки `a -> Either e b`, композиция через `>>=`
- `Left e` — **ноль** в этой категории (как `Nothing` в Maybe)
- Алгебраически: `Either e a ≅ e + a` — сумма типов

**Связь с Maybe:** `Maybe a ≅ Either () a`.
Заменяем «ничего» на конкретную единицу `()`.

**ExceptT** — трансформер, позволяющий встраивать Either в стек монад.

---
# 3️⃣ [] (List) — «Недетерминизм»

## 📐 Реализация

```haskell
instance Monad [] where
  return x = [x]          -- одно «возможное» значение
  xs >>= f = concatMap f xs  -- каждый элемент порождает список, склеиваем
```

**Ключевая идея:** список — это **недетерминированное вычисление**.
`[1,2,3]` означает «значение, которое *одновременно* равно 1, 2 и 3».
`>>=` применяет функцию к каждой «ветви» и собирает все результаты.

## ⭐ Главные функции

| Функция | Описание |
|---------|----------|
| `concatMap` | Основа `>>=` для списков |
| `guard` | Отфильтровать ветви (`mzero` при False) |
| `msum` | Первый непустой из списка списков |
| `mconcat` | Конкатенация через Monoid |
| `replicateM` | n независимых недетерминированных выборов |
| `sequence` | Список монадических действий → одно действие |

In [15]:
import Control.Monad (guard, replicateM)

-- concatMap — это и есть >>= для списков
[1,2,3] >>= \ x -> [x, x*10]
-- эквивалентно: concatMap (\ x -> [x, x*10]) [1,2,3]

[1,10,2,20,3,30]

In [16]:
-- guard: "убить" ветвь если условие не выполнено
-- guard True  = [()]   (ветвь живёт)
-- guard False = []     (ветвь умирает)

do
  x <- [1..10]
  guard (even x)
  guard (x `mod` 3 == 0)
  return x
-- числа от 1 до 10, кратные и 2, и 3

[6]

In [17]:
-- Пример: все перестановки
perms :: Eq a => [a] -> [[a]]
perms [] = [[]]
perms xs = do
  x    <- xs
  rest <- perms (filter (/= x) xs)
  return (x : rest)

-- Осторожно: [1..5] даёт 120 перестановок
length (perms [1..5 :: Int])

120

## 🛠️ Примеры использования

**Когда применять:** комбинаторика, поиск с перебором, генерация всех вариантов,
логическое программирование, решение задач типа N-ферзей.

In [18]:
-- Пример 1: Задача о рюкзаке (упрощённая)
items :: [(String, Int, Int)]  -- (название, вес, ценность)
items = [("книга",2,3),("ноутбук",5,10),("телефон",1,6),("планшет",3,7)]

-- Все подмножества
subsets :: [a] -> [[a]]
subsets []     = [[]]
subsets (x:xs) = let rest = subsets xs
                 in rest ++ map (x:) rest

-- Лучший набор до maxWeight кг
bestKnapsack :: Int -> [(String,Int,Int)] -> ([String], Int, Int)
bestKnapsack maxW its =
  let valid = [ (map (\ (n,_,_)->n) s, sum (map (\ (_,w,_)->w) s), sum (map (\ (_,_,v)->v) s))
              | s <- subsets its
              , sum (map (\ (_,w,_)->w) s) <= maxW ]
      best  = foldr (\ a@(_,_,v1) b@(_,_,v2) -> if v1 >= v2 then a else b) ([],0,0) valid
  in best

bestKnapsack 6 items

(["\1085\1086\1091\1090\1073\1091\1082","\1090\1077\1083\1077\1092\1086\1085"],6,16)

In [19]:
-- Пример 2: N-ферзей через List Monad
queens :: Int -> [[Int]]
queens n = go n
  where
    go 0 = [[]]
    go k = do
      qs <- go (k-1)
      q  <- [1..n]
      guard (safe q qs)
      return (q:qs)

    safe q qs = and [ q /= q'
                    && abs (q - q') /= abs (length qs - i)
                    | (i, q') <- zip [0..] qs ]

length (queens 8)   -- 92 решения для доски 8x8

0

In [20]:
-- Пример 3: replicateM — все бинарные строки длины n
replicateM 3 [0,1]  -- все 3-битные числа

[[0,0,0],[0,0,1],[0,1,0],[0,1,1],[1,0,0],[1,0,1],[1,1,0],[1,1,1]]

## 🔭 Категорная точка зрения

`[]` — это **монада мощности (power set monad)**:

- Объект `a` отображается в `P(a)` = `[a]` (множество элементов типа `a`)
- `return x = [x]` — синглтон, включение `a ↪ P(a)`
- `join = concat` — `P(P(a)) → P(a)`, объединение множества множеств
- `>>=` = `concatMap` = **flatMap** в других языках
- Это **монада в категории Rel** (бинарных отношений)

Список как монада моделирует **недетерминированные вычисления** — то же,
что в логическом программировании (Prolog) моделируется через бэктрекинг.

**MonadPlus:** `[]` — это также `MonadPlus` с `mzero = []` и `mplus = (++)`,
что делает его **монадой с выбором**.

---
# 4️⃣ IO — «Побочные эффекты под контролем»

## 📐 Реализация

`IO` — единственная монада в Haskell, реализованная **на уровне рантайма**,
а не как обычный Haskell-тип. Концептуально:

```haskell
-- IO — это функция из "состояния мира" в (результат, новое состояние мира)
newtype IO a = IO (RealWorld -> (a, RealWorld))

instance Monad IO where
  return x = IO (\ w -> (x, w))   -- не меняем мир, возвращаем x
  IO m >>= f = IO (\ w ->          -- выполняем m, передаём результат в f
    let (a, w') = m w
        IO m'   = f a
    in m' w')
```

**Ключевая идея:** `IO a` — это **рецепт** (description), а не немедленное действие.
Рантайм Haskell выполняет только `main :: IO ()`.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `putStr` / `putStrLn` | `String -> IO ()` | Вывод |
| `print` | `Show a => a -> IO ()` | Вывод через show |
| `getLine` | `IO String` | Ввод строки |
| `readLn` | `Read a => IO a` | Ввод и парсинг |
| `mapM` | `(a -> IO b) -> [a] -> IO [b]` | map для IO |
| `mapM_` | `(a -> IO ()) -> [a] -> IO ()` | map, игнорируя результаты |
| `forM` / `forM_` | Как mapM, но аргументы переставлены | |
| `when` / `unless` | Условное IO-действие | |
| `forever` | Бесконечный цикл IO | |
| `hSetBuffering` | Управление буферизацией | |

In [21]:
-- mapM_: выполнить IO для каждого элемента
mapM_ (\ (i,x) -> putStrLn $ show i ++ ". " ++ x)
  (zip [1..] ["Haskell", "ML", "Idris", "Agda"])

1. Haskell
2. ML
3. Idris
4. Agda

In [22]:
import Control.Monad (when, unless, forM_)

-- when: условное IO
let x = 42
when   (x > 40) $ putStrLn "x больше 40"
unless (x > 100) $ putStrLn "x не больше 100"

x больше 40

x не больше 100

In [23]:
-- forM_: как mapM_ но с переставленными аргументами (удобно для do)
import Control.Monad (forM_)

forM_ [1..5] $ \ i ->
  putStrLn $ "Квадрат " ++ show i ++ " = " ++ show (i*i)

Квадрат 1 = 1
Квадрат 2 = 4
Квадрат 3 = 9
Квадрат 4 = 16
Квадрат 5 = 25

## 🛠️ Примеры использования

**Когда применять:** любое взаимодействие с внешним миром — файлы, сеть,
консоль, случайность, время, изменяемые переменные (IORef).

In [24]:
-- Пример 1: IORef — мутабельная ссылка внутри IO
import Data.IORef

-- Простой счётчик через IORef
counter :: IO ()
counter = do
  ref <- newIORef (0 :: Int)
  forM_ [1..5] $ \ i -> modifyIORef ref (+i)
  val <- readIORef ref
  putStrLn $ "Сумма 1..5 = " ++ show val

counter

Сумма 1..5 = 15

In [25]:
-- Пример 2: sequence — выполнить список IO-действий
import Control.Monad (forM)

-- Собрать результаты нескольких вычислений
results <- forM [1..5] $ \ n -> do
  let sq = n * n
  return sq
putStrLn $ "Квадраты: " ++ show results

Квадраты: [1,4,9,16,25]

## 🔭 Категорная точка зрения

`IO` — это **монада состояния мира** (State RealWorld):

- Концептуально: `IO a ≅ State RealWorld a ≅ RealWorld -> (a, RealWorld)`
- `return` — тождественное преобразование мира
- `>>=` — последовательная композиция преобразований мира
- **Важно:** `RealWorld` — фантомный тип, гарантирующий линейность
  (мир нельзя скопировать или выбросить, только передать дальше)

**Уникальность IO:** в отличие от всех остальных монад, `IO` нельзя «запустить»
из чистого кода — только рантайм запускает `main`. Это гарантирует,
что **чистые функции не имеют побочных эффектов** (referential transparency).

`unsafePerformIO` существует, но нарушает все гарантии.

---
# 5️⃣ State s — «Изменяемое состояние без мутабельности»

## 📐 Реализация

```haskell
newtype State s a = State { runState :: s -> (a, s) }

instance Monad (State s) where
  return x   = State \ s -> (x, s)        -- вернуть x, состояние не трогать
  m >>= f    = State \ s ->
    let (a, s') = runState m s              -- запустить m, получить (результат, новое_состояние)
        (b, s'') = runState (f a) s'        -- передать результат в f, продолжить
    in (b, s'')
```

**Ключевая идея:** состояние передаётся **неявно** по всей цепочке.
Не нужно писать `s` в каждой функции вручную — монада делает это за тебя.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `get` | `State s s` | Прочитать состояние |
| `put` | `s -> State s ()` | Заменить состояние |
| `modify` | `(s->s) -> State s ()` | Изменить состояние |
| `modify'` | Строгая версия `modify` | Избегает утечек памяти |
| `gets` | `(s->a) -> State s a` | Прочитать проекцию |
| `runState` | `State s a -> s -> (a,s)` | Запустить, получить (результат, состояние) |
| `evalState` | `State s a -> s -> a` | Только результат |
| `execState` | `State s a -> s -> s` | Только финальное состояние |

In [26]:
import Control.Monad.State

-- Простой стек через State
type Stack = [Int]

push :: Int -> State Stack ()
push x = modify (x:)

pop :: State Stack Int
pop = do
  stack <- get
  case stack of
    []     -> error "empty stack"
    (x:xs) -> do { put xs; return x }

peek :: State Stack Int
peek = gets head

stackOps :: State Stack String
stackOps = do
  push 10
  push 20
  push 30
  a <- pop        -- 30
  b <- pop        -- 20
  push (a + b)    -- 50
  top <- peek
  return $ "Вершина стека: " ++ show top

runState stackOps []

("\1042\1077\1088\1096\1080\1085\1072 \1089\1090\1077\1082\1072: 50",[50,10])

In [27]:
-- Пример: генератор уникальных ID
import Control.Monad.State

type ID = Int

freshId :: State ID Int
freshId = do
  n <- get
  put (n + 1)
  return n

-- Создаём несколько узлов дерева с уникальными ID
data Node = Node Int String [Node] deriving (Show)

buildTree :: State ID Node
buildTree = do
  rootId <- freshId
  child1Id <- freshId
  child2Id <- freshId
  grandId  <- freshId
  let grand  = Node grandId  "внук"    []
      child1 = Node child1Id "ребёнок1" [grand]
      child2 = Node child2Id "ребёнок2" []
      root   = Node rootId   "корень"   [child1, child2]
  return root

evalState buildTree 0

Node 0 "\1082\1086\1088\1077\1085\1100" [Node 1 "\1088\1077\1073\1105\1085\1086\1082\&1" [Node 3 "\1074\1085\1091\1082" []],Node 2 "\1088\1077\1073\1105\1085\1086\1082\&2" []]

## 🔭 Категорная точка зрения

`State s` — это **монада для категории эндоморфизмов**:

- `State s a ≅ s -> (a, s)` — функция из состояния в пару
- Это **Reader и Writer одновременно**: читаем `s` и возвращаем новое `s`
- Категория Клейсли: стрелки `a -> State s b`
- **join:** `State s (State s a) -> State s a` — выполнить внешнее вычисление,
  затем внутреннее на полученном состоянии

`State s ≅ ReaderT s (Writer s)` — это одновременно Reader (читаем окружение)
и Writer (пишем новое окружение).

**Алгебра:** `State s` — это монада для **алгебры с изменяемой ячейкой памяти типа `s`**.

---
# 6️⃣ Reader r — «Разделяемое окружение»

## 📐 Реализация

```haskell
newtype Reader r a = Reader { runReader :: r -> a }

instance Monad (Reader r) where
  return x = Reader \ _ -> x     -- игнорируем окружение, возвращаем x
  m >>= f  = Reader \ r ->       -- передаём одно и то же окружение r везде
    let a = runReader m r
    in  runReader (f a) r
```

**Ключевая идея:** функции в цепочке автоматически получают одно и то же
**неизменяемое** окружение `r`. Не нужно передавать конфиг в каждую функцию.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `ask` | `Reader r r` | Получить всё окружение |
| `asks` | `(r->a) -> Reader r a` | Получить проекцию окружения |
| `local` | `(r->r) -> Reader r a -> Reader r a` | Запустить с изменённым окружением |
| `runReader` | `Reader r a -> r -> a` | Запустить с конкретным окружением |
| `reader` | `(r->a) -> Reader r a` | Конструктор |

In [28]:
import Control.Monad.Reader

-- Конфигурация приложения
data AppConfig = AppConfig
  { dbHost    :: String
  , dbPort    :: Int
  , appDebug  :: Bool
  , maxConns  :: Int
  } deriving (Show)

-- Функции получают конфиг неявно через Reader
getConnectionString :: Reader AppConfig String
getConnectionString = do
  host <- asks dbHost
  port <- asks dbPort
  return $ "postgresql://" ++ host ++ ":" ++ show port

logLevel :: Reader AppConfig String
logLevel = do
  debug <- asks appDebug
  return $ if debug then "DEBUG" else "INFO"

appInfo :: Reader AppConfig String
appInfo = do
  connStr  <- getConnectionString
  level    <- logLevel
  maxC     <- asks maxConns
  return $ unlines
    [ "Подключение: " ++ connStr
    , "Лог-уровень: " ++ level
    , "Макс. соединений: " ++ show maxC ]

cfg :: AppConfig
cfg = AppConfig { dbHost="localhost", dbPort=5432, appDebug=True, maxConns=10 }

putStr $ runReader appInfo cfg

Подключение: postgresql://localhost:5432
Лог-уровень: DEBUG
Макс. соединений: 10

In [29]:
-- local: запустить подвычисление с изменённым окружением
import Control.Monad.Reader

withTestDb :: Reader AppConfig a -> Reader AppConfig a
withTestDb = local $ \ cfg -> cfg { dbHost = "test-db", dbPort = 5433 }

prodConn :: Reader AppConfig String
prodConn = asks dbHost

testConn :: Reader AppConfig String
testConn = withTestDb (asks dbHost)

(runReader prodConn cfg, runReader testConn cfg)

("localhost","test-db")

## 🛠️ Примеры использования

**Когда применять:** dependency injection, конфигурация приложения,
передача окружения (логгер, БД, настройки) без явных аргументов.

In [30]:
-- Пример: Dependency Injection
import Control.Monad.Reader

data Env = Env
  { envLog  :: String -> IO ()
  , envFetch :: String -> IO String
  }

-- Программа зависит от окружения, но не знает конкретных реализаций
type App a = ReaderT Env IO a

fetchAndLog :: String -> App String
fetchAndLog url = do
  fetch <- asks envFetch
  logF  <- asks envLog
  liftIO $ logF $ "Fetching: " ++ url
  result <- liftIO $ fetch url
  liftIO $ logF $ "Got: " ++ take 20 result
  return result

-- Тестовое окружение (mock)
testEnv :: Env
testEnv = Env
  { envLog   = putStrLn . ("[LOG] " ++)
  , envFetch = \ url -> return $ "MOCK_DATA_FOR_" ++ url
  }

runReaderT (fetchAndLog "https://example.com") testEnv

[LOG] Fetching: https://example.com
[LOG] Got: MOCK_DATA_FOR_https:
"MOCK_DATA_FOR_https://example.com"

## 🔭 Категорная точка зрения

`Reader r` — это **монада для функций из фиксированного объекта**:

- `Reader r a ≅ r -> a` — просто функция!
- Категория Клейсли совпадает с обычной категорией функций из `r`
- `return x = const x` — константная функция
- `join :: Reader r (Reader r a) -> Reader r a` — диагональ: `join f r = f r r`
- Это **комонада** `(->)r` (функциональный тип как комонада)

**Связь:** `Reader r ≅ (->) r` — Reader — это просто функциональный тип в монадическом обёртке.
Все `Reader`-вычисления — это просто функции, монада лишь управляет их композицией.

---
# 7️⃣ Writer w — «Накопление лога»

## 📐 Реализация

```haskell
newtype Writer w a = Writer { runWriter :: (a, w) }

instance Monoid w => Monad (Writer w) where
  return x = Writer (x, mempty)    -- пустой лог
  m >>= f  = Writer $
    let (a, w1) = runWriter m      -- выполнить m, получить результат и лог
        (b, w2) = runWriter (f a)  -- передать результат в f
    in  (b, w1 <> w2)              -- СКЛЕИТЬ логи через (<>)
```

**Ключевая идея:** каждый шаг вычисления **накапливает** что-то в моноид `w`.
Это может быть: строка лога, список событий, сумма, любой `Monoid`.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `tell` | `w -> Writer w ()` | Добавить в лог |
| `listen` | `Writer w a -> Writer w (a, w)` | Получить лог вместе с результатом |
| `pass` | `Writer w (a, w->w) -> Writer w a` | Модифицировать лог |
| `runWriter` | `Writer w a -> (a, w)` | Получить пару |
| `execWriter` | `Writer w a -> w` | Только лог |
| `writer` | `(a, w) -> Writer w a` | Конструктор |

In [31]:
import Control.Monad.Writer

-- Логирование через Writer
loggedFactorial :: Integer -> Writer [String] Integer
loggedFactorial 0 = do
  tell ["factorial 0 = 1"]
  return 1
loggedFactorial n = do
  tell ["вычисляем factorial " ++ show n]
  prev <- loggedFactorial (n - 1)
  let result = n * prev
  tell ["factorial " ++ show n ++ " = " ++ show result]
  return result

let (result, logs) = runWriter (loggedFactorial 5)
putStrLn $ "Результат: " ++ show result
mapM_ putStrLn logs

Результат: 120

вычисляем factorial 5
вычисляем factorial 4
вычисляем factorial 3
вычисляем factorial 2
вычисляем factorial 1
factorial 0 = 1
factorial 1 = 1
factorial 2 = 2
factorial 3 = 6
factorial 4 = 24
factorial 5 = 120

In [32]:
-- Writer с числовым моноидом: подсчёт операций
import Control.Monad.Writer
import Data.Monoid (Sum(..))

countSteps :: [Int] -> Writer (Sum Int) [Int]
countSteps []     = return []
countSteps (x:xs) = do
  tell (Sum 1)           -- засчитываем шаг
  rest <- countSteps xs
  if even x
    then do tell (Sum 1) -- дополнительная операция для чётных
            return (x `div` 2 : rest)
    else return (x : rest)

let (result, Sum steps) = runWriter (countSteps [1,2,3,4,5,6])
putStrLn $ "Результат: " ++ show result
putStrLn $ "Шагов: " ++ show steps

Результат: [1,1,3,2,5,3]

Шагов: 9

## 🛠️ Примеры использования

**Когда применять:** логирование, аудит-трейл, сбор метрик/статистики,
накопление предупреждений, трассировка вычислений.

In [33]:
-- Пример: компилятор с накоплением предупреждений
import Control.Monad.Writer

data Warning = UnusedVar String | ShadowedVar String | TypeCoercion String
  deriving (Show)

type Compiler a = Writer [Warning] a

-- "Компиляция" выражений с предупреждениями
compileExpr :: String -> Compiler String
compileExpr "x" = do
  tell [UnusedVar "x"]
  return "reg_x"
compileExpr "y" = return "reg_y"
compileExpr expr = do
  tell [TypeCoercion expr]
  return $ "coerce(" ++ expr ++ ")"

compileFn :: [String] -> Compiler [String]
compileFn args = do
  regs <- mapM compileExpr args
  tell [ShadowedVar "temp"]
  return regs

let (regs, warnings) = runWriter (compileFn ["x", "y", "42"])
putStrLn $ "Регистры: " ++ show regs
putStrLn "Предупреждения:"
mapM_ (putStrLn . ("  ⚠ " ++) . show) warnings

Регистры: ["reg_x","reg_y","coerce(42)"]

Предупреждения:

  ⚠ UnusedVar "x"
  ⚠ TypeCoercion "42"
  ⚠ ShadowedVar "temp"

## 🔭 Категорная точка зрения

`Writer w` — это **монада для моноида**:

- Требует: `w` должен быть **моноидом** (`mempty` + `<>`)
- `Writer w a ≅ (a, w)` — просто пара!
- `return x = (x, mempty)` — результат с пустым логом
- `join :: Writer w (Writer w a) -> Writer w a` развернуть и склеить логи
- **Двойственность:** Writer двойственен Reader в смысле Category Theory:
  Reader читает окружение, Writer пишет в него

**Алгебра:** `Writer w` — это монада для **алгебры с накапливаемым эффектом**.
Тип `w` должен быть моноидом, иначе нельзя совместить «пустое» с «накопленным».

---
# 8️⃣ Identity — «Тривиальная монада»

## 📐 Реализация

```haskell
newtype Identity a = Identity { runIdentity :: a }

instance Monad Identity where
  return x = Identity x
  m >>= f  = f (runIdentity m)   -- просто применить функцию, никакого контекста
```

**Ключевая идея:** `Identity` — это монада, которая ничего не делает.
Она существует как **базовый случай** и **строительный блок** для трансформеров монад.

In [34]:
import Data.Functor.Identity

-- Identity просто заворачивает значение
let x = Identity 42
runIdentity x

-- Functор/Monad работают, но без эффектов
runIdentity $ do
  a <- Identity 10
  b <- Identity 20
  return (a + b)

42

30

In [35]:
-- Главный смысл: Identity как основа трансформеров
-- State s a == StateT s Identity a
-- Reader r a == ReaderT r Identity a
-- Writer w a == WriterT w Identity a

import Control.Monad.Trans.State (StateT, runStateT)
import Data.Functor.Identity

-- StateT Identity === State
type MyState s a = StateT s Identity a

-- Они изоморфны!
import Control.Monad.State (State, runState)

-- State  s a  ~  s -> (a, s)
-- StateT s Identity a  ~  s -> Identity (a,s) ~  s -> (a,s)

putStrLn "Identity — фундамент трансформеров монад"

Identity — фундамент трансформеров монад

## 🔭 Категорная точка зрения

`Identity` — это **тождественный эндофунктор**:

- В теории категорий: `Id : C -> C` — функтор, который всё оставляет как есть
- `return = Identity`, `join (Identity (Identity x)) = Identity x`
- Это **единица** для композиции монад через трансформеры

**Роль в трансформерах:** все монадные трансформеры `T` определяются так:
`T m a` для произвольной монады `m`. Когда `m = Identity`, получаем «чистую» монаду без трансформации.

---
# 9️⃣ Cont r — «Продолжения (Continuations)»

## 📐 Реализация

```haskell
newtype Cont r a = Cont { runCont :: (a -> r) -> r }
--                                   ^^^^^^^^   ^
--                               продолжение  финальный результат

instance Monad (Cont r) where
  return x = Cont (\ k -> k x)       -- передать x в продолжение
  m >>= f  = Cont (\ k ->
    runCont m (\ a ->                 -- запустить m с продолжением:
      runCont (f a) k))               -- передать результат в f, затем в k
```

**Ключевая идея:** вместо «вернуть результат», вычисление получает **функцию-продолжение**
(«что делать дальше») и само решает — вызывать ли её, когда, сколько раз.

`callCC` (call with current continuation) — «схватить» текущее продолжение.

## ⭐ Главные функции

| Функция | Тип | Описание |
|---------|-----|----------|
| `runCont` | `Cont r a -> (a->r) -> r` | Запустить с финальным продолжением |
| `evalCont` | `Cont a a -> a` | Запустить с `id` |
| `callCC` | `((a -> Cont r b) -> Cont r a) -> Cont r a` | Захватить продолжение |
| `cont` | `((a->r)->r) -> Cont r a` | Конструктор |
| `reset` | Ограничить область продолжения | |
| `shift` | Захватить делимитированное продолжение | |

In [36]:
import Control.Monad.Cont

-- Базовый пример: callCC как ранний выход
-- callCC позволяет «выпрыгнуть» из цепочки досрочно
earlyExit :: Int -> Cont String String
earlyExit n = callCC $ \ exit -> do
  when (n < 0)  $ exit "Ошибка: отрицательное число"
  when (n > 100)$ exit "Ошибка: слишком большое число"
  let result = n * 2
  return $ "Результат: " ++ show result

runCont (earlyExit 42)    id
-- runCont (earlyExit (-5)) id
-- runCont (earlyExit 200) id

"\1056\1077\1079\1091\1083\1100\1090\1072\1090: 84"

In [37]:
import Control.Monad.Cont

-- Cont как инверсия управления: «CPS-трансформация»
-- Обычная функция:
add :: Int -> Int -> Int
add x y = x + y

-- CPS-версия (continuation-passing style):
addCPS :: Int -> Int -> (Int -> r) -> r
addCPS x y k = k (x + y)

-- Через Cont:
addCont :: Int -> Int -> Cont r Int
addCont x y = cont (addCPS x y)

runCont (do
  a <- addCont 3 4
  b <- addCont a 10
  return b) id

Line 16: Redundant return
Found:
do a <- addCont 3 4
   b <- addCont a 10
   return b
Why not:
do a <- addCont 3 4
   addCont a 10

17

In [38]:
import Control.Monad.Cont

-- callCC для реализации исключений без IO
type Result = Either String Int

safeDivCont :: Int -> Int -> Cont Result Int
safeDivCont _ 0 = cont $ \ _ -> Left "деление на ноль"
safeDivCont x y = return (x `div` y)

computation :: Cont Result Int
computation = do
  a <- safeDivCont 100 5
  b <- safeDivCont a 2
  c <- safeDivCont b 0   -- тут упадём
  return (a + b + c)

runCont computation Right

Left "\1076\1077\1083\1077\1085\1080\1077 \1085\1072 \1085\1086\1083\1100"

## 🔭 Категорная точка зрения

`Cont r` — это **монада продолжений**, основанная на **двойной негации**:

- В логике Карри-Ховарда: `Cont r a ≅ (a -> r) -> r ≅ ¬¬a` (при `r = ⊥`)
- **Двойная негация** в логике: из `¬¬A` нельзя вывести `A` без закона исключённого третьего
- `callCC` соответствует **закону Пирса**: `((a -> b) -> a) -> a`
- Это монада **управления потоком** на уровне типов

**Универсальность Cont:** любую монаду можно закодировать через `Cont` (теорема Дандамиева).
`Cont` — это «мать всех монад».

---
# 🔟 ST s — «Безопасная мутабельность»

## 📐 Реализация

```haskell
-- ST s a — вычисление с мутабельным состоянием, параметризованное "токеном" s
newtype ST s a = ST (STRep s a)
type STRep s a = State# s -> (# State# s, a #)

-- Ключевые примитивы:
newSTRef   :: a          -> ST s (STRef s a)  -- создать ссылку
readSTRef  :: STRef s a  -> ST s a            -- прочитать
writeSTRef :: STRef s a  -> a -> ST s ()      -- записать
modifySTRef :: STRef s a -> (a->a) -> ST s () -- изменить

-- "Выход" из ST — только через runST:
runST :: (forall s. ST s a) -> a
```

**Ключевая идея:** тип `s` — это **фантомный тег**, который не позволяет
ссылкам «утекать» за пределы `runST`. Это гарантирует **referential transparency**
несмотря на внутреннюю мутабельность.

In [39]:
import Control.Monad.ST
import Data.STRef

-- Мутабельный аккумулятор внутри ST
sumST :: [Int] -> Int
sumST xs = runST $ do
  acc <- newSTRef 0
  mapM_ (\ x -> modifySTRef acc (+x)) xs
  readSTRef acc

sumST [1..100]

5050

In [40]:
import Control.Monad.ST
import Data.STRef
import Data.List (foldl')

-- Эффективная сортировка через ST (концептуально)
-- В реальном коде используют Data.Vector.Algorithms

-- Подсчёт вхождений через мутабельный Map (через список)
countOccurrences :: Ord a => [a] -> [(a,Int)]
countOccurrences xs = runST $ do
  ref <- newSTRef ([] :: [(a,Int)])
  mapM_ (\ x -> modifySTRef ref (insertOrUpdate x)) xs
  readSTRef ref
  where
    insertOrUpdate x [] = [(x,1)]
    insertOrUpdate x ((k,v):rest)
      | x == k    = (k, v+1) : rest
      | otherwise = (k,v) : insertOrUpdate x rest

countOccurrences "haskell is great and haskell is fun"

Line 12: Avoid lambda
Found:
\ x -> modifySTRef ref (insertOrUpdate x)
Why not:
modifySTRef ref . insertOrUpdate

[('h',2),('a',4),('s',4),('k',2),('e',3),('l',4),(' ',6),('i',2),('g',1),('r',1),('t',1),('n',2),('d',1),('f',1),('u',1)]

## 🛠️ Примеры использования

**Когда применять:** производительные алгоритмы (сортировка, парсинг),
мутабельные массивы (`Data.Array.ST`), граф-алгоритмы — всё где нужна
эффективность мутабельности с гарантиями чистоты.

In [41]:
import Control.Monad.ST
import Data.Array.ST
import Data.Array

-- Алгоритм: заполнение массива через STArray
buildArray :: Int -> Array Int Int
buildArray n = runSTArray $ do
  arr <- newArray (0, n-1) 0
  mapM_ (\ i -> writeArray arr i (i*i)) [0..n-1]
  return arr

buildArray 10

array (0,9) [(0,0),(1,1),(2,4),(3,9),(4,16),(5,25),(6,36),(7,49),(8,64),(9,81)]

## 🔭 Категорная точка зрения

`ST s` — это **монада состояния с параметризованным регионом**:

- Ранговый 2 полиморфизм: `runST :: (∀s. ST s a) -> a`
  означает «для всех возможных тегов s» — никакого конкретного `s` не "утечёт"
- **Тип s** — это **линейный ресурс** в смысле линейной логики: создаётся один раз, не копируется
- Аналог в линейных типах: `s` — "владелец" мутабельных ссылок, и он не может выйти из скоупа
- `ST s ≅ IO` по внутренней реализации — `IO = ST RealWorld`

**Теорема Войта-Лаунчбери:** `runST` типобезопасен благодаря ранговому полиморфизму.

---
# 1️⃣1️⃣ STM — «Транзакционная память»

## 📐 Реализация

```haskell
-- STM a — транзакционное вычисление, возвращающее a
newtype STM a = STM (State# RealWorld -> (# State# RealWorld, a #))

-- Примитивы:
newTVar   :: a -> STM (TVar a)        -- создать транзакционную переменную
readTVar  :: TVar a -> STM a          -- прочитать
writeTVar :: TVar a -> a -> STM ()    -- записать
retry     :: STM a                    -- откатить и подождать изменений
orElse    :: STM a -> STM a -> STM a  -- попробовать первое, при retry — второе

-- Запуск транзакции:
atomically :: STM a -> IO a           -- выполнить атомарно
```

**Ключевая идея:** `STM` — транзакционная память.
Изменения либо применяются **все**, либо **откатываются** (как в БД).
**retry** блокирует транзакцию до изменения хотя бы одной читаемой `TVar`.

## ⭐ Главные функции

| Функция | Описание |
|---------|----------|
| `newTVar` / `readTVar` / `writeTVar` | Работа с TVar в STM |
| `newTVarIO` | Создать TVar прямо в IO |
| `modifyTVar` / `modifyTVar'` | Изменить TVar |
| `retry` | Откатить, ждать изменений |
| `orElse` | Альтернатива при retry |
| `check` | `retry` если условие ложно |
| `TMVar` | TVar + Maybe (блокирующий канал) |
| `TQueue` | Транзакционная очередь |
| `TChan` | Транзакционный канал (broadcast) |

In [42]:
import Control.Concurrent.STM
import Control.Concurrent (forkIO, threadDelay)
import Control.Monad (replicateM_, forM_)

-- Классика: перевод денег между счетами
type Account = TVar Int

transfer :: Account -> Account -> Int -> STM ()
transfer from to amount = do
  fromBal <- readTVar from
  toBal   <- readTVar to
  if fromBal < amount
    then retry  -- недостаточно средств — ждать
    else do
      writeTVar from (fromBal - amount)
      writeTVar to   (toBal   + amount)

-- Демо (без реального параллелизма для Jupyter)
demo :: IO ()
demo = do
  alice <- newTVarIO 1000
  bob   <- newTVarIO 500

  atomically $ transfer alice bob 300

  aliceBal <- readTVarIO alice
  bobBal   <- readTVarIO bob
  putStrLn $ "Alice: " ++ show aliceBal
  putStrLn $ "Bob:   " ++ show bobBal

demo

Alice: 700
Bob:   800

In [43]:
import Control.Concurrent.STM

-- TMVar: блокирующий MVar на основе STM
-- TQueue: транзакционная FIFO очередь

mailbox :: IO ()
mailbox = do
  q <- newTQueueIO

  -- "Отправляем" сообщения
  atomically $ do
    writeTQueue q "Привет"
    writeTQueue q "Мир"
    writeTQueue q "Haskell"

  -- "Читаем" сообщения
  msgs <- atomically $ do
    m1 <- readTQueue q
    m2 <- readTQueue q
    m3 <- readTQueue q
    return [m1, m2, m3]

  mapM_ putStrLn msgs

mailbox

Привет
Мир
Haskell

## 🔭 Категорная точка зрения

`STM` — это **монада транзакций с откатом**:

- Транзакция = набор операций чтения/записи `TVar`
- `retry` + `orElse` делают `STM` **MonadPlus**: `mzero = retry`, `mplus = orElse`
- **Семантика**: `atomically t` запускает `t` оптимистично (без блокировок),
  при конфликте — откатывает и повторяет (MVCC: Multi-Version Concurrency Control)
- Это **монада с эффектом «управления временем»**: `retry` останавливает выполнение
  до изменения наблюдаемых переменных

**Гарантии:** отсутствие дедлоков (нет явных блокировок), композабельность (транзакции можно вкладывать через `orElse`).

---
# 1️⃣2️⃣ Parser — «Самодельный парсер как монада»

## 📐 Реализация

```haskell
-- Парсер: берёт строку, возвращает (результат, остаток) или неудачу
newtype Parser a = Parser { runParser :: String -> Maybe (a, String) }

instance Monad Parser where
  return x = Parser \ input -> Just (x, input)  -- успех, не consume ничего
  p >>= f  = Parser \ input ->
    case runParser p input of
      Nothing       -> Nothing                -- первый парсер не сработал
      Just (a, rest) -> runParser (f a) rest  -- передаём остаток строки дальше
```

**Ключевая идея:** монада позволяет **последовательно** потреблять входную строку,
каждый шаг получает «хвост» от предыдущего.

In [44]:
-- Полная реализация Parser Monad с нуля
import Data.Char (isDigit, isAlpha, isSpace)

newtype Parser a = Parser { runParser :: String -> Maybe (a, String) }

instance Functor Parser where
  fmap f p = Parser $ \input ->
    case runParser p input of
      Nothing        -> Nothing
      Just (a, rest) -> Just (f a, rest)

instance Applicative Parser where
  pure x = Parser $ \input -> Just (x, input)
  pf <*> pa = Parser $ \input ->
    case runParser pf input of
      Nothing        -> Nothing
      Just (f, rest) -> runParser (fmap f pa) rest

instance Monad Parser where
  return = pure
  p >>= f = Parser $ \input ->
    case runParser p input of
      Nothing        -> Nothing
      Just (a, rest) -> runParser (f a) rest

-- Базовые парсеры
item :: Parser Char
item = Parser $ \input -> case input of
  []     -> Nothing
  (c:cs) -> Just (c, cs)

satisfy :: (Char -> Bool) -> Parser Char
satisfy p = Parser $ \input -> case input of
  []     -> Nothing
  (c:cs) -> if p c then Just (c, cs) else Nothing

char :: Char -> Parser Char
char c = satisfy (== c)

digit :: Parser Char
digit = satisfy isDigit

letter :: Parser Char
letter = satisfy isAlpha

-- Тест
runParser (char 'H') "Hello"

Line 28: Use lambda-case
Found:
\ input
  -> case input of
       [] -> Nothing
       (c : cs) -> Just (c, cs)
Why not:
\case
  [] -> Nothing
  (c : cs) -> Just (c, cs)Line 33: Use lambda-case
Found:
\ input
  -> case input of
       [] -> Nothing
       (c : cs) -> if p c then Just (c, cs) else Nothing
Why not:
\case
  [] -> Nothing
  (c : cs) -> if p c then Just (c, cs) else Nothing

Just ('H',"ello")

In [45]:
-- Продолжение: many и парсинг чисел
import Data.Char (digitToInt)

many :: Parser a -> Parser [a]
many p = Parser $ \input ->
  case runParser p input of
    Nothing        -> Just ([], input)
    Just (a, rest) ->
      case runParser (many p) rest of
        Nothing          -> Just ([a], rest)
        Just (as, rest') -> Just (a:as, rest')

natural :: Parser Int
natural = Parser $ \input ->
  case runParser (many (satisfy isDigit)) input of
    Nothing       -> Nothing
    Just ([], _)  -> Nothing
    Just (ds, rest) -> Just (read ds, rest)

integer :: Parser Int
integer = Parser $ \input ->
  case input of
    ('-':rest) -> fmap (\(n,r) -> (-n, r)) (runParser natural rest)
    _          -> runParser natural input

runParser integer "-42 rest"

Just (-42," rest")

In [46]:
-- sepBy и финальный тест
spaces :: Parser ()
spaces = Parser $ \input ->
  Just ((), dropWhile isSpace input)

sepBy :: Parser a -> Parser sep -> Parser [a]
sepBy p sep = Parser $ \input ->
  case runParser p input of
    Nothing -> Just ([], input)
    Just (a, rest) ->
      case runParser sep rest of
        Nothing        -> Just ([a], rest)
        Just (_, rest2) ->
          case runParser (sepBy p sep) rest2 of
            Nothing          -> Just ([a], rest)
            Just (as, rest3) -> Just (a:as, rest3)

-- Парсим "1,2,3,4,5"
runParser (sepBy natural (char ',')) "1,2,3,4,5"

Just ([1,2,3,4,5],"")

## 🔭 Категорная точка зрения

Parser — это **монада над Maybe-обёртками над потребляемым потоком**:

- `Parser a ≅ String -> Maybe (a, String)`
- Это частный случай монады **State с Maybe**: `StateT String Maybe a`
- `<|>` (Alternative) + `many`/`some` = структура для бэктрекинга
- **Теорема**: Parser = комбинатор парсеров = монада + альтернатива + ещё примитивы

**Связь с реальными библиотеками:** `parsec`, `megaparsec`, `attoparsec` —
все построены по этому принципу, только с лучшей диагностикой ошибок и производительностью.

---
# 1️⃣3️⃣ Free f — «Свободная монада / DSL»

## 📐 Реализация

```haskell
data Free f a
  = Pure a           -- чистый результат (листья)
  | Free (f (Free f a))  -- узел: один шаг f, содержащий продолжение

instance Functor f => Monad (Free f) where
  return = Pure
  Pure a  >>= f = f a
  Free fx >>= f = Free (fmap (>>= f) fx)  -- проталкиваем f вглубь
```

**Ключевая идея:** `Free f` — это дерево, где:
- Листья = `Pure a` (финальный результат)
- Узлы = `f (...)" (шаги вашего DSL)
- Монадная структура позволяет **строить** дерево через do-нотацию
- **Интерпретатор** потом обходит это дерево и выполняет команды

Отделяем **описание программы** от её **выполнения**.

In [47]:
-- Free монада: реализация с нуля (без внешних пакетов)
data Free f a
  = Pure a
  | Free (f (Free f a))

instance Functor f => Functor (Free f) where
  fmap f (Pure a)  = Pure (f a)
  fmap f (Free fx) = Free (fmap (fmap f) fx)

instance Functor f => Applicative (Free f) where
  pure = Pure
  Pure f  <*> x = fmap f x
  Free ff <*> x = Free (fmap (<*> x) ff)

instance Functor f => Monad (Free f) where
  return = Pure
  Pure a  >>= f = f a
  Free fx >>= f = Free (fmap (>>= f) fx)

-- Поднять один шаг f в Free
liftF :: Functor f => f a -> Free f a
liftF fa = Free (fmap Pure fa)

-- ШАГ 1: DSL для консоли
data ConsoleF next
  = PrintLine String next
  | ReadLine  (String -> next)
  deriving (Functor)

type Console a = Free ConsoleF a

-- Smart constructors
printLine :: String -> Console ()
printLine s = liftF (PrintLine s ())

readLine :: Console String
readLine = liftF (ReadLine id)

-- Программа-описание (ещё не выполнена)
program :: Console ()
program = do
  printLine "Как тебя зовут?"
  name <- readLine
  printLine $ "Привет, " ++ name ++ "!"
  printLine "Haskell Free Monad работает!"

putStrLn "DSL программа построена (ещё не выполнена)"

DSL программа построена (ещё не выполнена)

In [48]:
-- Два интерпретатора для одной программы

-- Интерпретатор 1: pure тест (без IO, с готовыми ответами)
interpretPure :: [String] -> Console a -> (a, [String])
interpretPure _      (Pure x)                   = (x, [])
interpretPure inputs (Free (PrintLine s next))  =
  let (a, out) = interpretPure inputs next
  in  (a, s : out)
interpretPure (i:is) (Free (ReadLine  f))       = interpretPure is (f i)
interpretPure []     (Free (ReadLine  f))       = interpretPure [] (f "")

-- Интерпретатор 2: реальный IO
interpretIO :: Console a -> IO a
interpretIO (Pure x)                  = return x
interpretIO (Free (PrintLine s next)) = putStrLn s >> interpretIO next
interpretIO (Free (ReadLine  f))      = return "Alice" >>= \line -> interpretIO (f line)

-- Тестируем pure-интерпретатором (без IO!)
let (_, outputs) = interpretPure ["Alice"] program
mapM_ putStrLn (reverse outputs)

Haskell Free Monad работает!
Привет, Alice!
Как тебя зовут?

In [49]:
-- Более реальный пример: DSL для работы с БД
import qualified Data.Map.Strict as Map

data DbF next
  = Select String ([String] -> next)
  | Insert String String next
  | Delete String next
  deriving (Functor)

type Db a = Free DbF a

-- Smart constructors
dbSelect :: String -> Db [String]
dbSelect t = liftF (Select t id)

dbInsert :: String -> String -> Db ()
dbInsert t v = liftF (Insert t v ())

dbDelete :: String -> Db ()
dbDelete t = liftF (Delete t ())

-- Программа на нашем DSL
dbProgram :: Db [String]
dbProgram = do
  dbInsert "users" "Alice"
  dbInsert "users" "Bob"
  dbDelete "temp"
  dbSelect "users"

-- In-memory интерпретатор
type MockDb = Map.Map String [String]

interpretMock :: MockDb -> Db a -> (a, MockDb)
interpretMock db (Pure x)                    = (x, db)
interpretMock db (Free (Insert t v next))    =
  interpretMock (Map.insertWith (++) t [v] db) next
interpretMock db (Free (Delete t next))      =
  interpretMock (Map.delete t db) next
interpretMock db (Free (Select t f))         =
  let rows = Map.findWithDefault [] t db
  in  interpretMock db (f rows)

let (result, finalDb) = interpretMock Map.empty dbProgram
putStrLn $ "SELECT users: " ++ show result
putStrLn $ "Финальная БД: " ++ show finalDb

SELECT users: ["Bob","Alice"]

Финальная БД: fromList [("users",["Bob","Alice"])]

## 🔭 Категорная точка зрения

`Free f` — это **левый сопряжённый к забывающему функтору**:

- Для любого функтора `f` и монады `m`: `Free f` — **наименьшая монада, содержащая `f`**
- Универсальное свойство: морфизм `f ~> m` (natural transformation) единственным образом
  поднимается до морфизма монад `Free f ~> m` (т.е. интерпретатора!)
- **Это и есть суть:** интерпретатор = natural transformation `f ~> m`

```
Functor f  →  Free f  →  интерпретатор  →  конкретная монада m
```

**Operational monad, Freer monad** — варианты Free с лучшей производительностью (через `GADTs`).
**Eff** — библиотека эффектов на основе этой идеи.

---
# 1⃣4⃣ Q (Template Haskell) — «Монада цитирования»

## 📐 Реализация

`Q` — монада из пакета `template-haskell`, предоставляемая компилятором GHC.
Она работает **во время компиляции**, а не во время выполнения.

```haskell
-- Концептуальная реализация (Q внутри GHC недоступна):
newtype Q a = Q { unQ :: QState -> IO (a, QState) }

data QState = QState
  { qFresh  :: Int           -- счётчик для генерации уникальных имён
  , qReify  :: Name -> Info  -- инфо о типах/функциях
  , qLookup :: String -> Maybe Name
  }

instance Monad Q where
  return x = Q $ \s -> return (x, s)
  m >>= f  = Q $ \s -> do
    (a, s') <- unQ m s
    unQ (f a) s'
```

**Ключевая идея:** `Q` — это монада состояния поверх IO, выполняемая **компилятором**.
Умеет: генерировать имена (`newName`), цитировать AST (`[| ... |]`), интроспекцию (`reify`).

## 📐 Типы AST в Template Haskell

```haskell
-- Основные типы AST:
data Exp  = VarE Name          -- переменная
          | AppE Exp Exp       -- применение функции
          | LamE [Pat] Exp     -- лямбда
          | LitE Lit           -- литерал
          | InfixE ...         -- инфиксный оператор
          | DoE [Stmt]         -- do-блок
          | ...

data Type = VarT Name          -- типовая переменная
          | AppT Type Type     -- применение типового конструктора
          | ConT Name          -- типовой конструктор (Int, Maybe, ...)
          | ArrowT             -- стрелочка ->
          | ForallT ...        -- полиморфный тип

data Dec  = FunD Name [Clause] -- объявление функции
          | DataD ...          -- тип данных
          | SigD Name Type     -- аннотация типа
          | InstanceD ...

-- Квазицитаты: удобный синтаксис для построения AST:
-- [e| expr |]  :: Q Exp      -- цитата выражения
-- [t| type |]  :: Q Type     -- цитата типа
-- [d| decls |] :: Q [Dec]    -- цитата объявлений
-- [p| pat   |] :: Q Pat      -- цитата паттерна
```

## ⭐ Главные функции монады Q

| Функция | Тип | Описание |
|---------|-----|----------|
| `runQ` | `Q a -> IO a` | Запустить Q в IO (для отладки) |
| `newName` | `String -> Q Name` | Свежее уникальное имя |
| `reify` | `Name -> Q Info` | Инфо о типе/функции |
| `reifyInstances` | `Name -> [Type] -> Q [InstanceDec]` | Найти инстансы |
| `lookupTypeName` | `String -> Q (Maybe Name)` | Найти имя типа |
| `location` | `Q Loc` | Текущее место в файле |
| `isInstance` | `Name -> [Type] -> Q Bool` | Проверить инстанс |
| `[e\| ... \|]` | `Q Exp` | Цитата выражения |
| `[t\| ... \|]` | `Q Type` | Цитата типа |
| `[d\| ... \|]` | `Q [Dec]` | Цитата объявлений |
| `pprint` | `Ppr a => a -> String` | Красивая печать AST |
| `runIO` | `IO a -> Q a` | IO во время компиляции |

In [50]:
import Language.Haskell.TH

-- В IHaskell квазицитаты [e|..|] недоступны.
-- Покажем AST выражения 1 + 2 вручную:
let ast = InfixE
            (Just (LitE (IntegerL 1)))
            (VarE (mkName "+"))
            (Just (LitE (IntegerL 2)))
print ast
putStrLn $ pprint ast

InfixE (Just (LitE (IntegerL 1))) (VarE +) (Just (LitE (IntegerL 2)))

1 + 2

In [51]:
import Language.Haskell.TH
-- AST лямбды \x -> x * 2
let x = mkName "x"
    ast = LamE [VarP x]
               (InfixE (Just (VarE x))
                       (VarE (mkName "*"))
                       (Just (LitE (IntegerL 2))))
print ast
putStrLn $ "«" ++ pprint ast ++ "»"

LamE [VarP x] (InfixE (Just (VarE x)) (VarE *) (Just (LitE (IntegerL 2))))

«\x -> x * 2»

In [52]:
import Language.Haskell.TH
-- AST типа: Int -> Maybe String
let ast = AppT (AppT ArrowT (ConT (mkName "Int")))
               (AppT (ConT (mkName "Maybe")) (ConT (mkName "String")))
print ast
putStrLn $ pprint ast

AppT (AppT ArrowT (ConT Int)) (AppT (ConT Maybe) (ConT String))

Int -> Maybe String

In [53]:
import Language.Haskell.TH
-- AST объявления: double x = x * 2
let x   = mkName "x"
    ast = FunD (mkName "double")
               [ Clause [VarP x]
                        (NormalB (InfixE (Just (VarE x))
                                         (VarE (mkName "*"))
                                         (Just (LitE (IntegerL 2)))))
                        [] ]
putStrLn $ pprint ast

double x = x * 2

## newName — генерация уникальных имён

`newName` генерирует имена вида `x_0`, `x_1`, ... чтобы избежать конфликтов
при генерации кода. Это **гигиеническое** метапрограммирование: сгенерированные имена
не защитывают переменные из окружающего кода.

In [54]:
import Language.Haskell.TH

-- Генерируем несколько свежих имён
runQ $ do
  a <- newName "x"
  b <- newName "x"    -- то же базовое, но другой суффикс
  c <- newName "acc"
  return [show a, show b, show c]

["x_0","x_1","acc_2"]

## Построение AST вручную

Квазицитаты (`[e| ... |]`) удобны, но иногда нужно строить AST программно.
Монадический интерфейс `Q` позволяет делать это чисто и безопасно.

In [55]:
import Language.Haskell.TH

-- Построим AST \x -> x + 1 вручную:
x <- runQ $ newName "x"
let manual = LamE [VarP x]
                  (InfixE (Just (VarE x))
                           (VarE (mkName "+"))
                           (Just (LitE (IntegerL 1))))
putStrLn $ "AST:    " ++ show manual
putStrLn $ "pprint: " ++ pprint manual

AST:    LamE [VarP x_3] (InfixE (Just (VarE x_3)) (VarE +) (Just (LitE (IntegerL 1))))

pprint: \x_0 -> x_0 + 1

## reify — интроспекция во время компиляции

`reify` позволяет получить информацию о **любом** имени: функции, типе, классе,
конструкторе — прямо во время компиляции. Это основа для derive-like генерации кода.

In [56]:
import Language.Haskell.TH

-- reify с lookupValueName работает только в компилируемых файлах, не в GHCi
-- В реальном коде это выглядело бы так:
-- runQ (reify 'map) или runQ $ do { Just n <- lookupValueName "map"; reify n }

-- Покажем типы Info напрямую:
putStrLn "VarI name type        -- для функции"
putStrLn "TyConI dec            -- для типа"
putStrLn "ClassI dec instances  -- для класса"
putStrLn "DataConI name type    -- для конструктора"

VarI name type        -- для функции

TyConI dec            -- для типа

ClassI dec instances  -- для класса

DataConI name type    -- для конструктора

In [57]:
import Language.Haskell.TH

-- Пример результата reify ''Maybe (TyConI):
putStrLn "TyConI (DataD [] Maybe [PlainTV a ()] Nothing"
putStrLn "        [NormalC Nothing [], NormalC Just [(Bang NoSourceUnpackedness"
putStrLn "                                                NoSourceStrictness, VarT a)]]"
putStrLn "        [DerivClause Nothing []])"
putStrLn ""
putStrLn "структура AST типа Maybe: 2 конструктора (Nothing и Just)"

TyConI (DataD [] Maybe [PlainTV a ()] Nothing

        [NormalC Nothing [], NormalC Just [(Bang NoSourceUnpackedness

                                                NoSourceStrictness, VarT a)]]

        [DerivClause Nothing []])

структура AST типа Maybe: 2 конструктора (Nothing и Just)

In [58]:
import Language.Haskell.TH

-- Пример результата reify ''Num (ClassI):
putStrLn "ClassI (ClassD [AppT (ConT Eq) (VarT a)] Num ..."
putStrLn "        [ SigD fromInteger :: Integer -> a"
putStrLn "        , SigD (+) :: a -> a -> a"
putStrLn "        , SigD (-) :: a -> a -> a"
putStrLn "        , SigD (*) :: a -> a -> a"
putStrLn "        , SigD negate :: a -> a"
putStrLn "        , SigD abs    :: a -> a"
putStrLn "        , SigD signum :: a -> a ]) [instance Num Int, ...]"
putStrLn "класс Num содержит 7 методов"

ClassI (ClassD [AppT (ConT Eq) (VarT a)] Num ...

        [ SigD fromInteger :: Integer -> a

        , SigD (+) :: a -> a -> a

        , SigD (-) :: a -> a -> a

        , SigD (*) :: a -> a -> a

        , SigD negate :: a -> a

        , SigD abs    :: a -> a

        , SigD signum :: a -> a ]) [instance Num Int, ...]

класс Num содержит 7 методов

## 🛠️ Практический пример 1: генерация геттеров

Классический use-case TH — генерация boilerplate.
`makeGetters ''MyType` сгенерирует getter-функцию для каждого поля record-типа.

In [59]:
import Language.Haskell.TH

-- Генератор геттеров для record-типа
makeGetters :: Name -> Q [Dec]
makeGetters typeName = do
  info <- reify typeName
  case info of
    TyConI (DataD _ _ _ _ [RecC _ fields] _) ->
      mapM makeGetter fields
    _ -> fail $ "makeGetters: " ++ show typeName ++ " не record-тип"
  where
    makeGetter (fieldName, _, fieldType) = do
      argName <- newName "r"
      let getterName = mkName $ "get_" ++ nameBase fieldName
          body       = NormalB $ AppE (VarE fieldName) (VarE argName)
          clause     = Clause [VarP argName] body []
          sig        = SigD getterName
                            (AppT (AppT ArrowT (ConT typeName)) fieldType)
      return $ FunD getterName [clause]

putStrLn "Генератор геттеров определён"

Генератор геттеров определён

In [60]:
import Language.Haskell.TH

-- Тестируем makeGetters: сделаем Point внутри runQ
decs <- runQ $ do
  -- Вручную строим AST типа Point и генерируем геттеры
  let pointX = mkName "pointX"
      pointY = mkName "pointY"
      point  = mkName "Point"
  xGetter <- newName "r"
  yGetter <- newName "r"
  return
    [ FunD (mkName "get_pointX")
        [Clause [VarP xGetter] (NormalB (AppE (VarE pointX) (VarE xGetter))) []]
    , FunD (mkName "get_pointY")
        [Clause [VarP yGetter] (NormalB (AppE (VarE pointY) (VarE yGetter))) []]
    ]
mapM_ (putStrLn . pprint) decs

get_pointX r_0 = pointX r_0
get_pointY r_0 = pointY r_0

## 🛠️ Практический пример 2: генерация по шаблону

Q позволяет генерировать произвольное число функций по параметру.
Сгенерируем геттер i-го элемента из кортежа любого размера.

In [61]:
import Language.Haskell.TH

-- nthTupleGetter size idx: геттер idx-го элемента из кортежа размера size
nthTupleGetter :: Int -> Int -> Q Exp
nthTupleGetter size idx = do
  names <- mapM (\i -> newName ("t" ++ show i)) [0..size-1]
  let pat  = TupP (map VarP names)
      body = VarE (names !! idx)
  return $ LamE [pat] body

-- Геттер 2-го элемента из четвёрки
getter <- runQ (nthTupleGetter 4 2)
putStrLn $ "pprint: " ++ pprint getter
putStrLn $ "AST:    " ++ show getter

pprint: \(t0_0, t1_1, t2_2, t3_3) -> t2_2

AST:    LamE [TupP [VarP t0_6,VarP t1_7,VarP t2_8,VarP t3_9]] (VarE t2_8)

## 🛠️ Практический пример 3: проверки на этапе компиляции

TH позволяет **встроить** проверки в компиляцию.
Ошибка проявится не в рантайме, а ещё при сборке.

In [62]:
import Language.Haskell.TH

-- Функция проверяет во время компиляции что список непустой
staticHead :: Q Exp -> Q Exp
staticHead listQ = do
  listExpr <- listQ
  case listExpr of
    ListE []    -> fail "Ошибка: staticHead получил пустой список!"
    ListE (x:_) -> return x
    _           -> fail "Не литеральный список"

-- Передаём AST списка вручную (=== [10,20,30]):
let listAst = return $ ListE [LitE (IntegerL 10), LitE (IntegerL 20), LitE (IntegerL 30)]
result <- runQ $ staticHead listAst
putStrLn $ "Первый элемент: " ++ pprint result

Первый элемент: 10

In [63]:
import Language.Haskell.TH

-- Провальный случай — вызвал бы ошибку при компиляции:
-- runQ $ staticHead [e| [] |]

-- Проверка идентификатора: валидные символы
staticIdent :: String -> Q Exp
staticIdent s
  | null s             = fail "Пустая строка"
  | not (all ok s)     = fail $ "Невалидный символ в '" ++ s ++ "'"
  | otherwise          = return $ LitE (StringL s)
  where ok c = c `elem` (['a'..'z'] ++ ['A'..'Z'] ++ ['0'..'9'] ++ "_'")

runQ $ staticIdent "validName_123"
-- runQ $ staticIdent "invalid name!"  -- ← пало бы при компиляции

LitE (StringL "validName_123")

## 🛠️ Практический пример 4: генерация числового перечисления

Генерируем `data` объявление и вспомогательные функции (`toInt`, `fromInt`, `allValues`)
программно, по списку имён конструкторов.

In [64]:
import Language.Haskell.TH

-- mkEnum: генерируем ADT из списка имён конструкторов
mkEnum :: String -> [String] -> Q [Dec]
mkEnum typeName conNames = do
  let tName   = mkName typeName
      cons    = map (\n -> NormalC (mkName n) []) conNames
      derives = [ DerivClause Nothing
                    [ ConT (mkName "Show"), ConT (mkName "Eq")
                    , ConT (mkName "Ord"),  ConT (mkName "Bounded")
                    , ConT (mkName "Enum") ] ]
      dataDec = DataD [] tName [] Nothing cons derives
      allName = mkName ("all" ++ typeName)
      allBody = NormalB $ ListE (map (ConE . mkName) conNames)
      allDec  = FunD allName [Clause [] allBody []]
      allSig  = SigD allName (AppT ListT (ConT tName))
  return [dataDec, allSig, allDec]

-- Посмотрим сгенерированный код:
runQ (mkEnum "Weekday" ["Mon","Tue","Wed","Thu","Fri"]) >>= mapM_ (putStrLn . pprint)

data Weekday
    = Mon
    | Tue
    | Wed
    | Thu
    | Fri
    deriving (Show, Eq, Ord, Bounded, Enum)
allWeekday :: [Weekday]
allWeekday = [Mon, Tue, Wed, Thu, Fri]

## QuasiQuoters — расширение синтаксиса

`QuasiQuoter` позволяет создать **свой синтаксис** в Haskell через `[myParser| ... |]`.

```haskell
data QuasiQuoter = QuasiQuoter
  { quoteExp  :: String -> Q Exp
  , quotePat  :: String -> Q Pat
  , quoteType :: String -> Q Type
  , quoteDec  :: String -> Q [Dec]
  }
```

Примеры из реальных библиотек:
- `[r| raw string \n |]` — сырые строки без экранирования
- `[sql| SELECT * FROM users |]` — SQL в коде
- `[json| {"key": 1} |]` — JSON литералы
- `[regex| \d+\.\d+ |]` — регулярки с compile-time проверкой
- `[hamlet| <div>#{name}</div> |]` — HTML шаблоны (Yesod)

In [65]:
import Language.Haskell.TH
import Language.Haskell.TH.Quote

-- Простой QuasiQuoter: обращает строку задом
revStr :: QuasiQuoter
revStr = QuasiQuoter
  { quoteExp  = \s -> return $ LitE (StringL (reverse s))
  , quotePat  = \s -> return $ LitP (StringL (reverse s))
  , quoteType = \_ -> fail "revStr: не применим к типам"
  , quoteDec  = \_ -> fail "revStr: не применим к объявлениям"
  }

-- [revStr| olleH dlrow |]  ==  "Hello world"
-- Демо через runQ:
result <- runQ $ quoteExp revStr "olleH dlrow"
putStrLn $ show result

LitE (StringL "world Hello")

In [66]:
import Language.Haskell.TH
import Language.Haskell.TH.Quote
import Data.Char (isAlphaNum)

-- QuasiQuoter для простого двумерного формата: "key=value, key2=value2"
-- В результате получаем [("key","value"),("key2","value2")]
pairs :: QuasiQuoter
pairs = QuasiQuoter
  { quoteExp  = \s ->
      let kvs  = map parsePair (splitOn ',' s)
          parsePair p = case break (== '=') p of
            (k, '=':v) -> TupE [Just (LitE (StringL (trim k))),
                                  Just (LitE (StringL (trim v)))]
            _          -> error $ "bad pair: " ++ p
          trim = reverse . dropWhile (== ' ') . reverse . dropWhile (== ' ')
          splitOn _ [] = [[]]
          splitOn c (x:xs)
            | x == c    = [] : splitOn c xs
            | otherwise = let (h:t) = splitOn c xs in (x:h):t
      in return $ ListE kvs
  , quotePat  = \_ -> fail "pairs: no Pat"
  , quoteType = \_ -> fail "pairs: no Type"
  , quoteDec  = \_ -> fail "pairs: no Dec"
  }

result <- runQ $ quoteExp pairs " host=localhost , port=5432 , db=mydb "
putStrLn $ pprint result

[("host", "localhost"), ("port", "5432"), ("db", "mydb")]

## runIO — IO во время компиляции

`runIO :: IO a -> Q a` позволяет выполнять IO-действия **во время сборки**:
читать файлы, читать переменные окружения, встраивать данные в бинарник.

In [67]:
import Language.Haskell.TH
import Data.Time (getCurrentTime)

-- Встраиваем время сборки в код как константу:
buildTimestamp :: Q Exp
buildTimestamp = do
  t <- runIO getCurrentTime
  return $ LitE (StringL (show t))

-- В реальном коде: buildVersion = $(buildTimestamp)
-- Автоматически встроено в бинарник!
runQ buildTimestamp

LitE (StringL "2026-06-01 14:46:42.614418558 UTC")

## Сплайсинг: `$(...)` и `''`, `'`

```haskell
{-# LANGUAGE TemplateHaskell #-}

-- Цитата имён и сплайсинг:
'myFunc      -- Name функции myFunc  (VarE 'myFunc)
''MyType     -- Name типа   MyType  (ConT ''MyType)

-- Сплайс выражения (Exp Splice):
let x = $(someQExp)          -- вставить сгенерированное выражение

-- Сплайс объявления (Dec Splice):
$(makeGetters ''Person)      -- сгенерировать функции на верхнем уровне

-- Типизированный сплайс (Typed Splice, GHC 9+):
let y = $$(someTypedQExp :: TExp Int)

-- Ограничения:
-- 1. Стадийное ограничение: splice не может использовать
--    определения из того же модуля
-- 2. TH-код компилируется отдельной фазой
```

In [68]:
import Language.Haskell.TH

-- Цитаты имён ('name, ''Type) работают только в файлах с TemplateHaskell
-- В GHCi/IHaskell используем mkName:
let mapName   = mkName "map"
    maybeName = mkName "Maybe"
    numName   = mkName "Num"
    justName  = mkName "Just"
putStrLn $ "Name функции map:  " ++ show mapName
putStrLn $ "Name типа   Maybe: " ++ show maybeName
putStrLn $ "Name класса  Num:   " ++ show numName
putStrLn $ "Name констр. Just:  " ++ show justName
putStrLn ""
putStrLn "В реальном коде с TemplateHaskell:"
putStrLn "  'map      -- Name функции (VarE 'map)"
putStrLn "  ''Maybe   -- Name типа   (ConT ''Maybe)"

Name функции map:  map

Name типа   Maybe: Maybe

Name класса  Num:   Num

Name констр. Just:  Just

В реальном коде с TemplateHaskell:

  'map      -- Name функции (VarE 'map)

  ''Maybe   -- Name типа   (ConT ''Maybe)

## 🔭 Категорная точка зрения

`Q` — это **монада состояния компилятора**:

- `Q a ≅ QState -> IO (a, QState)` — это `StateT QState IO`
  (читаем окружение компилятора + IO для reify)

**Двухуровневая семантика (Two-Level Types):**
```
Уровень 0 (runtime):  обычный Haskell код
Уровень 1 (compile): Q-вычисления, цитаты, сплайсы
```

**Связь с модальной логикой S4 (Дэвис-Пфеннинг, 1996):**

| Логика S4 | Template Haskell |
|------------|------------------|
| □A (необходимо A) | `Q Exp` — код типа A |
| ◇A (возможно A) | `$(splice)` — выполнить |
| η: A → □A | `[e\| ... \|]` — цитировать |
| ε: □A → A | `$(...)` — сплайсинг |

Пара (`[e| ... |]`, `$(...)`) образует **монадическое сопряжение (adjunction)**:
`Quote ⊣ Splice`, что делает их **функторами между категориями** метауровня и объектного уровня.

**Связь с другими монадами:**
- `Q ≅ StateT QState IO` — по внутренней структуре
- `Q` — юникальна тем, что нельзя вызвать `runQ` в обычном коде — только компилятор запускает её
  (аналогично `IO` и `main`)

## 📚 Реальные библиотеки, использующие Q

| Библиотека | Что использует TH |
|------------|-------------------|
| `lens` | `makeLenses` — генерация линз для record-полей |
| `aeson` | `deriveJSON` — генерация To/FromJSON |
| `persistent` | `mkPersist` — ORM, генерация схемы БД |
| `yesod` | `hamlet`, `julius` — HTML/JS квазицитаты |
| `servant` | Генерация клиентов/серверов из типа API |
| `file-embed` | Встраивание файлов в бинарник |
| `regex-quasiquoter` | Compile-time проверка регулярок |
| `th-lift` | Автоматический `Lift` для произвольных типов |

```haskell
-- lens:
data Person = Person { _name :: String, _age :: Int }
makeLenses ''Person
-- генерирует: name :: Lens' Person String
--              age  :: Lens' Person Int

-- aeson:
data Config = Config { host :: String, port :: Int }
deriveJSON defaultOptions ''Config
-- генерирует: instance ToJSON Config
--              instance FromJSON Config
```

---
# 🔗 Связи между монадами

## 15.1 Иерархия мощности

```
Identity          — нет эффектов, базовый случай
    │
    ├── Maybe         — провал без данных
    │       └── Either e   — провал с данными
    │
    ├── []            — недетерминизм
    │
    ├── IO            — произвольные эффекты (вершина)
    │       ├── ST s       — мутабельность (подмножество IO)
    │       │       └── STM    — транзакции (надстройка над IO)
    │       └── Q          — IO во время компиляции (runIO)
    │
    ├── State s       — изменяемое состояние
    │       ≅ Reader s + Writer s
    ├── Reader r      — окружение (функция r->a)
    └── Writer w      — логирование (пара (a,w))
```

```
Free f            — свободная монада (строитель DSL)
Cont r            — продолжения (кодирует любую монаду)
Parser            ≅ StateT String Maybe
Q                 ≅ StateT QState IO  (монада компилятора)
```

### Диаграмма сравнения монад по мощности

<svg width="600" height="340" xmlns="http://www.w3.org/2000/svg" font-family="monospace" font-size="13">
  <!-- Background -->
  <rect width="600" height="340" fill="#1e1e2e" rx="10"/>
  <!-- Title -->
  <text x="300" y="28" text-anchor="middle" fill="#cdd6f4" font-size="15" font-weight="bold">Иерархия мощности монад</text>
  <!-- Arrows (bottom to top = increasing power) -->
  <!-- Identity -->
  <rect x="220" y="290" width="160" height="30" rx="5" fill="#313244"/>
  <text x="300" y="310" text-anchor="middle" fill="#a6e3a1">Identity</text>
  <!-- Maybe -->
  <rect x="100" y="235" width="120" height="30" rx="5" fill="#313244"/>
  <text x="160" y="255" text-anchor="middle" fill="#89b4fa">Maybe</text>
  <line x1="220" y1="300" x2="160" y2="265" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Either -->
  <rect x="380" y="235" width="120" height="30" rx="5" fill="#313244"/>
  <text x="440" y="255" text-anchor="middle" fill="#89b4fa">Either e</text>
  <line x1="380" y1="300" x2="440" y2="265" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- List -->
  <rect x="240" y="175" width="120" height="30" rx="5" fill="#313244"/>
  <text x="300" y="195" text-anchor="middle" fill="#f5c2e7">[ ] (List)</text>
  <line x1="160" y1="235" x2="270" y2="205" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- State -->
  <rect x="60" y="115" width="120" height="30" rx="5" fill="#313244"/>
  <text x="120" y="135" text-anchor="middle" fill="#fab387">State s</text>
  <!-- Reader -->
  <rect x="200" y="115" width="120" height="30" rx="5" fill="#313244"/>
  <text x="260" y="135" text-anchor="middle" fill="#fab387">Reader r</text>
  <!-- Writer -->
  <rect x="340" y="115" width="120" height="30" rx="5" fill="#313244"/>
  <text x="400" y="135" text-anchor="middle" fill="#fab387">Writer w</text>
  <!-- IO -->
  <rect x="200" y="55" width="200" height="30" rx="5" fill="#45475a"/>
  <text x="300" y="75" text-anchor="middle" fill="#f38ba8" font-weight="bold">IO (вершина)</text>
  <!-- Connections List->State/Reader/Writer -->
  <line x1="300" y1="175" x2="120" y2="145" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="300" y1="175" x2="260" y2="145" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="300" y1="175" x2="400" y2="145" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- State/Reader/Writer -> IO -->
  <line x1="120" y1="115" x2="250" y2="85" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="260" y1="115" x2="285" y2="85" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="400" y1="115" x2="350" y2="85" stroke="#6c7086" stroke-width="1.5" marker-end="url(#arr)"/>
  <!-- Arrowhead marker -->
  <defs><marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
    <path d="M0,0 L0,6 L8,3 z" fill="#6c7086"/>
  </marker></defs>
</svg>

**Читать**: стрелка A -> B означает "B мощнее A" (B может выразить больше эффектов).
Identity < Maybe < Either < List < State/Reader/Writer < IO.


## 15.2 Изоморфизмы и включения

| Монада A | ≅ / ⊂ | Монада B | Примечание |
|----------|--------|----------|------------|
| `Maybe a` | `≅` | `Either () a` | Left () = Nothing |
| `State s a` | `≅` | `s -> (a, s)` | Функция состояния |
| `Reader r a` | `≅` | `r -> a` | Просто функция |
| `Writer w a` | `≅` | `(a, w)` | Просто пара |
| `State s a` | `≅` | `ReaderT s (Writer s) a` | Reader + Writer |
| `Parser a` | `≅` | `StateT String Maybe a` | State + Maybe |
| `ST s a` | `≅` | `State RealWorld a` | Концептуально |
| `IO a` | `≅` | `ST RealWorld a` | По реализации GHC |
| `Identity a` | `≅` | `a` | Тривиально |
| `Cont r a` | кодирует | любую монаду | Universality |
| `Q a` | `≅` | `StateT QState IO a` | Монада компилятора |
| `Free f a` | свободна над | любым функтором `f` | Левый сопряжённый |

In [69]:
-- Демо: Maybe ≅ Either ()
fromMaybe' :: Maybe a -> Either () a
fromMaybe' Nothing  = Left ()
fromMaybe' (Just x) = Right x

toMaybe' :: Either () a -> Maybe a
toMaybe' (Left ())  = Nothing
toMaybe' (Right x)  = Just x

fromMaybe' (Just 42)
-- fromMaybe' Nothing
-- toMaybe' (Left ())

Right 42

In [70]:
-- Демо: State ≅ Reader + Writer
import Control.Monad.State
import Control.Monad.RWS

-- RWS = Reader + Writer + State в одной монаде!
type App a = RWS
  String    -- r: окружение (конфиг)
  [String]  -- w: лог
  Int       -- s: состояние (счётчик)
  a

appLogic :: App String
appLogic = do
  cfg     <- ask               -- читаем конфиг
  n       <- get               -- читаем состояние
  tell ["шаг " ++ show n]      -- пишем в лог
  modify (+1)                  -- меняем состояние
  tell ["конфиг: " ++ cfg]
  return $ "результат-" ++ show n

let (result, finalState, logs) = runRWS appLogic "prod-config" 0
putStrLn $ "Результат: " ++ result
putStrLn $ "Состояние: " ++ show finalState
putStrLn $ "Лог: "       ++ show logs

Результат: результат-0

Состояние: 1

Лог: ["\1096\1072\1075 0","\1082\1086\1085\1092\1080\1075: prod-config"]

## 15.3 Законы монад (для всех монад!)

```haskell
-- 1. Левая единица (Left Identity):
return a >>= f   ≡   f a

-- 2. Правая единица (Right Identity):
m >>= return     ≡   m

-- 3. Ассоциативность (Associativity):
(m >>= f) >>= g  ≡   m >>= (\x -> f x >>= g)
```

Или через Клейсли-композицию `(>=>)`:
```haskell
-- 1. return >=> f      ≡  f                   (левая единица)
-- 2. f >=> return      ≡  f                   (правая единица)
-- 3. (f >=> g) >=> h   ≡  f >=> (g >=> h)     (ассоциативность)
```
Это делает категорию Клейсли **категорией** в математическом смысле.

In [71]:
-- Проверка законов монады для Maybe
import Control.Monad ((>=>))

f :: Int -> Maybe Int
f x = if x > 0 then Just (x * 2) else Nothing

g :: Int -> Maybe Int
g x = if even x then Just (x + 1) else Nothing

-- Закон 1: return a >>= f ≡ f a
law1 = (return 5 >>= f) == f 5

-- Закон 2: m >>= return ≡ m
law2 = (Just 5 >>= return) == Just 5

-- Закон 3: (m >>= f) >>= g ≡ m >>= (\x -> f x >>= g)
law3 = ((Just 5 >>= f) >>= g) == (Just 5 >>= (\ x -> f x >>= g))

putStrLn $ "Закон 1 (left identity):  " ++ show law1
putStrLn $ "Закон 2 (right identity): " ++ show law2
putStrLn $ "Закон 3 (associativity):  " ++ show law3

Line 11: Monad law, left identity
Found:
return 5 >>= f
Why not:
f 5Line 14: Monad law, right identity
Found:
Just 5 >>= return
Why not:
Just 5Line 17: Use >=>
Found:
\ x -> f x >>= g
Why not:
f >=> g

Закон 1 (left identity):  True

Закон 2 (right identity): True

Закон 3 (associativity):  True

## 15.4 Когда какую монаду выбрать?

![Kogda kakuyu monadu](../diagrams/monads/monad_choice.svg)

---
## 🧩Итоговая диаграмма

![Иерархия монад](../diagrams/monads/monad_hierarchy.svg)

**Трансформеры монад** (следующий ноутбук): MaybeT, ExceptT, StateT, ReaderT, WriterT, ContT — позволяют **комбинировать** эффекты нескольких монад в одном стеке.

---

**← Предыдущий:** [Foldable & Traversable](FoldableTraversable.ipynb)  |  **Следующий →** [Трансформеры монад](MonadTransformers.ipynb)
